In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
# 1. Import Libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# 2. Load Data
train= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [4]:
#column with highest missing value
print(train.isnull().sum().sort_values(ascending=False).head())

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
dtype: int64


In [5]:
# 3. Drop Columns
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']

train = train.drop(cols_to_drop, axis=1)
test = test.drop(cols_to_drop, axis=1)

In [6]:
# 4. Separate Features & Target
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

In [7]:
#5. Missing Value Handling
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Numeric -> Median
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

# Categorical -> Missing
X[cat_cols] = X[cat_cols].fillna('Missing')
test[cat_cols] = test[cat_cols].fillna('Missing')

#Verify missing value
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [8]:
#6.One-Hot Encode
X = pd.get_dummies(X)
test_encoded = pd.get_dummies(test)

X, test_encoded = X.align(
    test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

print(X.shape)
print(test_encoded.shape)

(1460, 287)
(1459, 287)


In [9]:
#7. Log Transform Target
y= np.log1p(y)
print(y.head())

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [10]:
# 8. Train/Validation Split
X_train,X_valid,y_train,y_valid= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_valid.shape)

(1168, 287)
(292, 287)


In [11]:
#9. Train Model
from sklearn.ensemble import RandomForestRegressor

model=RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train,y_train)

RandomForestRegressor(max_depth=20, n_jobs=-1, random_state=42)

# Run Cross Validation

In [12]:
from sklearn.model_selection import cross_val_score

scores= cross_val_score(model,
                       X,y,
                       cv=5,
                       scoring="neg_root_mean_squared_error")

In [13]:
#Convert Back
rmse_scores = -scores

print(rmse_scores)

print("Average RMSE:", rmse_scores.mean())
print("Standard Deviation:", rmse_scores.std())

[0.13922482 0.15572539 0.1435514  0.12992636 0.14947968]
Average RMSE: 0.14358152869059124
Standard Deviation: 0.008805758984102482


In [14]:
# 10. Validation Prediction
valid_pred=model.predict(X_valid)

#valid_pred= np.expm1(valid_pred)
print(valid_pred[0:5])

[11.85651821 12.68207754 11.65911675 11.94945621 12.65399041]


In [15]:
# 11. RMSE
rmse= root_mean_squared_error(y_valid,valid_pred)
print("RMSE:", rmse)

RMSE: 0.14374922924984443


In [16]:
# 12. Train on Full Data
model.fit(X, y)

RandomForestRegressor(max_depth=20, n_jobs=-1, random_state=42)

In [17]:
# 13. Predict Test Data
prediction = model.predict(test_encoded)


#Convert Back to Real Prices
prediction = np.expm1(prediction)
print(prediction)

[123597.54150494 153469.88761794 179476.07688604 ... 150503.02411191
 112388.35452537 234866.08314887]


In [18]:
print(prediction[:10])

[123597.54150494 153469.88761794 179476.07688604 183970.58877694
 194339.31990027 183844.21090635 160585.94248187 177011.7938921
 183272.9806426  121153.56354325]


In [19]:
#14. Submission File
submission=pd.DataFrame({
    'Id':test['Id'],
    'SalePrice': prediction
})

submission.to_csv('submission_v6.csv',index=False)

submission.head()

,Id,SalePrice
0,1461,123597.541505
1,1462,153469.887618
2,1463,179476.076886
3,1464,183970.588777
4,1465,194339.319900
